In [27]:
# Medicare Provider Analytics Using Apache Spark

## Dataset Profiling

# Course: CS-675 Big Data Management & Analytics

# Author: Judi-Ann Beckford

# Objective: Explore and profile the Medicare Provider Service and Enrollment datasets before data integration.

In [28]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

In [29]:
spark = (
    SparkSession.builder
    .appName("Medicare Dataset Profiling")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.maxResultSize", "2g")
    .getOrCreate()
)

print("Spark Version:", spark.version)

Spark Version: 4.1.2


In [30]:
provider_file = "../data/raw/PHY_R26_P05_V10_D24_Prov_Svc.csv"

enrollment_file = "../data/raw/PPEF_Enrollment_Extract_2026.04.01.csv"

In [31]:
print("Provider exists:", os.path.exists(provider_file))
print("Enrollment exists:", os.path.exists(enrollment_file))
print("Current folder:", os.getcwd())

Provider exists: True
Enrollment exists: True
Current folder: /Users/judi-annbeckford/Documents/Medicare-Provider-Analytics-Spark/notebooks


In [32]:
provider_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(provider_file)
)

print("Provider dataset loaded.")

[Stage 18:========================================>               (18 + 6) / 25]

Provider dataset loaded.


In [33]:
print("Rows:", provider_df.count())
print("Columns:", len(provider_df.columns))

[Stage 19:========================================>               (18 + 6) / 25]

Rows: 9781673
Columns: 28


In [4]:
provider_df.show(5, truncate=False)

+----------+--------------------------+-----------------------+---------------+--------------------+-------------------+-------------------------+----------------+-----------------+-------------------------+-----------------------+-----------------+-----------------+-----------------------------------------------------------------------------------+------------------+-----------------+-----------------------------+--------+----------------------------------------------------------------------------------------------------------------------------------+--------------+-------------+---------+---------+------------------+--------------+------------------+-----------------+------------------+
|Rndrng_NPI|Rndrng_Prvdr_Last_Org_Name|Rndrng_Prvdr_First_Name|Rndrng_Prvdr_MI|Rndrng_Prvdr_Crdntls|Rndrng_Prvdr_Ent_Cd|Rndrng_Prvdr_St1         |Rndrng_Prvdr_St2|Rndrng_Prvdr_City|Rndrng_Prvdr_State_Abrvtn|Rndrng_Prvdr_State_FIPS|Rndrng_Prvdr_Zip5|Rndrng_Prvdr_RUCA|Rndrng_Prvdr_RUCA_Desc             

In [34]:
provider_df.printSchema()

root
 |-- Rndrng_NPI: integer (nullable = true)
 |-- Rndrng_Prvdr_Last_Org_Name: string (nullable = true)
 |-- Rndrng_Prvdr_First_Name: string (nullable = true)
 |-- Rndrng_Prvdr_MI: string (nullable = true)
 |-- Rndrng_Prvdr_Crdntls: string (nullable = true)
 |-- Rndrng_Prvdr_Ent_Cd: string (nullable = true)
 |-- Rndrng_Prvdr_St1: string (nullable = true)
 |-- Rndrng_Prvdr_St2: string (nullable = true)
 |-- Rndrng_Prvdr_City: string (nullable = true)
 |-- Rndrng_Prvdr_State_Abrvtn: string (nullable = true)
 |-- Rndrng_Prvdr_State_FIPS: string (nullable = true)
 |-- Rndrng_Prvdr_Zip5: string (nullable = true)
 |-- Rndrng_Prvdr_RUCA: double (nullable = true)
 |-- Rndrng_Prvdr_RUCA_Desc: string (nullable = true)
 |-- Rndrng_Prvdr_Cntry: string (nullable = true)
 |-- Rndrng_Prvdr_Type: string (nullable = true)
 |-- Rndrng_Prvdr_Mdcr_Prtcptg_Ind: string (nullable = true)
 |-- HCPCS_Cd: string (nullable = true)
 |-- HCPCS_Desc: string (nullable = true)
 |-- HCPCS_Drug_Ind: string (nullable 

In [35]:
provider_df.show(10, truncate=False)

+----------+--------------------------+-----------------------+---------------+--------------------+-------------------+-------------------------+-----------------+-----------------+-------------------------+-----------------------+-----------------+-----------------+-----------------------------------------------------------------------------------+------------------+-----------------+-----------------------------+--------+----------------------------------------------------------------------------------------------------------------------------------+--------------+-------------+---------+---------+------------------+--------------+------------------+-----------------+------------------+
|Rndrng_NPI|Rndrng_Prvdr_Last_Org_Name|Rndrng_Prvdr_First_Name|Rndrng_Prvdr_MI|Rndrng_Prvdr_Crdntls|Rndrng_Prvdr_Ent_Cd|Rndrng_Prvdr_St1         |Rndrng_Prvdr_St2 |Rndrng_Prvdr_City|Rndrng_Prvdr_State_Abrvtn|Rndrng_Prvdr_State_FIPS|Rndrng_Prvdr_Zip5|Rndrng_Prvdr_RUCA|Rndrng_Prvdr_RUCA_Desc           

In [7]:
provider_df.select(
    "Rndrng_NPI",
    "Rndrng_Prvdr_Type",
    "HCPCS_Cd",
    "HCPCS_Desc",
    "Tot_Srvcs",
    "Avg_Mdcr_Pymt_Amt"
).show(10, truncate=False)

+----------+-----------------+--------+----------------------------------------------------------------------------------------------------------------------------------+---------+-----------------+
|Rndrng_NPI|Rndrng_Prvdr_Type|HCPCS_Cd|HCPCS_Desc                                                                                                                        |Tot_Srvcs|Avg_Mdcr_Pymt_Amt|
+----------+-----------------+--------+----------------------------------------------------------------------------------------------------------------------------------+---------+-----------------+
|1003000126|Internal Medicine|99221   |Initial hospital care with straightforward or low level of medical decision making, per day, if using time, at least 40 minutes   |36       |60.828888889     |
|1003000126|Internal Medicine|99222   |Initial hospital care with straightforward or low-level medical decision making, if using time, at least 55 minutes               |150      |95.6756          |
|1003

In [36]:
enrollment_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(enrollment_file)
)

print("Enrollment dataset loaded.")

[Stage 24:==============================================>         (20 + 4) / 24]

Enrollment dataset loaded.


In [37]:
print("Rows:", enrollment_df.count())
print("Columns:", len(enrollment_df.columns))

Rows: 2981799
Columns: 11


In [38]:
enrollment_df.printSchema()

root
 |-- NPI: integer (nullable = true)
 |-- MULTIPLE_NPI_FLAG: string (nullable = true)
 |-- PECOS_ASCT_CNTL_ID: long (nullable = true)
 |-- ENRLMT_ID: string (nullable = true)
 |-- PROVIDER_TYPE_CD: string (nullable = true)
 |-- PROVIDER_TYPE_DESC: string (nullable = true)
 |-- STATE_CD: string (nullable = true)
 |-- FIRST_NAME: string (nullable = true)
 |-- MDL_NAME: string (nullable = true)
 |-- LAST_NAME: string (nullable = true)
 |-- ORG_NAME: string (nullable = true)



In [39]:
enrollment_df.show(10, truncate=False)

+----------+-----------------+------------------+---------------+----------------+---------------------------------------+--------+----------+--------+---------+--------------------------------+
|NPI       |MULTIPLE_NPI_FLAG|PECOS_ASCT_CNTL_ID|ENRLMT_ID      |PROVIDER_TYPE_CD|PROVIDER_TYPE_DESC                     |STATE_CD|FIRST_NAME|MDL_NAME|LAST_NAME|ORG_NAME                        |
+----------+-----------------+------------------+---------------+----------------+---------------------------------------+--------+----------+--------+---------+--------------------------------+
|1003000126|N                |7517003643        |I20091005000100|14-11           |PRACTITIONER - INTERNAL MEDICINE       |MD      |ARDALAN   |NULL    |ENKESHAFI|NULL                            |
|1003000126|N                |7517003643        |I20130530000085|14-C6           |PRACTITIONER - HOSPITALIST             |DC      |ARDALAN   |NULL    |ENKESHAFI|NULL                            |
|1003000126|N            

In [40]:
provider_df.select([
    F.count(
        F.when(
            F.col(c).isNull(),
            c
        )
    ).alias(c)
    for c in provider_df.columns
]).show()

[Stage 29:========================================>               (18 + 6) / 25]

+----------+--------------------------+-----------------------+---------------+--------------------+-------------------+----------------+----------------+-----------------+-------------------------+-----------------------+-----------------+-----------------+----------------------+------------------+-----------------+-----------------------------+--------+----------+--------------+-------------+---------+---------+------------------+--------------+------------------+-----------------+------------------+
|Rndrng_NPI|Rndrng_Prvdr_Last_Org_Name|Rndrng_Prvdr_First_Name|Rndrng_Prvdr_MI|Rndrng_Prvdr_Crdntls|Rndrng_Prvdr_Ent_Cd|Rndrng_Prvdr_St1|Rndrng_Prvdr_St2|Rndrng_Prvdr_City|Rndrng_Prvdr_State_Abrvtn|Rndrng_Prvdr_State_FIPS|Rndrng_Prvdr_Zip5|Rndrng_Prvdr_RUCA|Rndrng_Prvdr_RUCA_Desc|Rndrng_Prvdr_Cntry|Rndrng_Prvdr_Type|Rndrng_Prvdr_Mdcr_Prtcptg_Ind|HCPCS_Cd|HCPCS_Desc|HCPCS_Drug_Ind|Place_Of_Srvc|Tot_Benes|Tot_Srvcs|Tot_Bene_Day_Srvcs|Avg_Sbmtd_Chrg|Avg_Mdcr_Alowd_Amt|Avg_Mdcr_Pymt_Amt|Avg_

In [41]:
enrollment_df.select([
    F.count(
        F.when(
            F.col(c).isNull(),
            c
        )
    ).alias(c)
    for c in enrollment_df.columns
]).show()

[Stage 32:==============================>                         (13 + 6) / 24]

+---+-----------------+------------------+---------+----------------+------------------+--------+----------+--------+---------+--------+
|NPI|MULTIPLE_NPI_FLAG|PECOS_ASCT_CNTL_ID|ENRLMT_ID|PROVIDER_TYPE_CD|PROVIDER_TYPE_DESC|STATE_CD|FIRST_NAME|MDL_NAME|LAST_NAME|ORG_NAME|
+---+-----------------+------------------+---------+----------------+------------------+--------+----------+--------+---------+--------+
|  0|                0|                 0|        0|               0|                 0|       0|    433496| 1402772|   433496| 2548303|
+---+-----------------+------------------+---------+----------------+------------------+--------+----------+--------+---------+--------+



In [44]:
provider_df.dtypes

[('Rndrng_NPI', 'int'),
 ('Rndrng_Prvdr_Last_Org_Name', 'string'),
 ('Rndrng_Prvdr_First_Name', 'string'),
 ('Rndrng_Prvdr_MI', 'string'),
 ('Rndrng_Prvdr_Crdntls', 'string'),
 ('Rndrng_Prvdr_Ent_Cd', 'string'),
 ('Rndrng_Prvdr_St1', 'string'),
 ('Rndrng_Prvdr_St2', 'string'),
 ('Rndrng_Prvdr_City', 'string'),
 ('Rndrng_Prvdr_State_Abrvtn', 'string'),
 ('Rndrng_Prvdr_State_FIPS', 'string'),
 ('Rndrng_Prvdr_Zip5', 'string'),
 ('Rndrng_Prvdr_RUCA', 'double'),
 ('Rndrng_Prvdr_RUCA_Desc', 'string'),
 ('Rndrng_Prvdr_Cntry', 'string'),
 ('Rndrng_Prvdr_Type', 'string'),
 ('Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'string'),
 ('HCPCS_Cd', 'string'),
 ('HCPCS_Desc', 'string'),
 ('HCPCS_Drug_Ind', 'string'),
 ('Place_Of_Srvc', 'string'),
 ('Tot_Benes', 'int'),
 ('Tot_Srvcs', 'double'),
 ('Tot_Bene_Day_Srvcs', 'int'),
 ('Avg_Sbmtd_Chrg', 'double'),
 ('Avg_Mdcr_Alowd_Amt', 'double'),
 ('Avg_Mdcr_Pymt_Amt', 'double'),
 ('Avg_Mdcr_Stdzd_Amt', 'double')]

In [46]:
provider_df.select("Rndrng_NPI").distinct().count()

1207473

In [47]:
enrollment_df.select("NPI").distinct().count()

2556656

In [48]:
provider_df.groupBy("Rndrng_NPI").count().filter("count > 1").count()

1067955

In [49]:
enrollment_df.groupBy("NPI").count().filter("count > 1").count()

281965

In [43]:
## # Dataset Summary

# - The Provider dataset contains nearly 9.8 million Medicare provider-service records.
# - The Enrollment dataset contains nearly 3 million provider enrollment records.
# - Both datasets share the NPI field.
# - These datasets share the National Provider Identifier (NPI), which will be used as the primary key for integration in the next stage of the project.